In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

# ===== 1) 读取数据并建立词表 =====
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print('词表大小:', vocab_size)
print('训练集长度:', len(train_data), '验证集长度:', len(val_data))

词表大小: 65
训练集长度: 1003854 验证集长度: 111540


In [12]:
# ===== 2) 批次采样函数 =====
batch_size = 32
block_size = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split):
    """
    从 train/val 中随机截取 batch_size 条长度为 block_size 的片段。
    返回:
    - x: (B, T)
    - y: (B, T)，对应 x 右移一位后的监督目标
    """
    source = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('x 形状:', xb.shape, 'y 形状:', yb.shape, 'device:', device)

x 形状: torch.Size([32, 8]) y 形状: torch.Size([32, 8]) device: cpu


## 一、Bigram 模型：训练与生成

In [13]:
class BigramLanguageModel(nn.Module):
    """
    Bigram 参数化方式：
    - token id -> 对应行 logits
    - 使用 Embedding(vocab_size, vocab_size) 实现高效查表
    """
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) -> logits: (B, T, C)
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # 每步只根据最后一个 token 预测下一个 token
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits_last = logits[:, -1, :]
            probs = F.softmax(logits_last, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

bigram_model = BigramLanguageModel(vocab_size).to(device)
logits, loss = bigram_model(xb, yb)
print('Bigram logits 形状:', logits.shape, 'loss:', float(loss))

Bigram logits 形状: torch.Size([256, 65]) loss: 4.692231178283691


In [14]:
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    """
    在 train/val 上估计平均损失，便于观察训练趋势。
    """
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def train_model(model, max_iters=400, lr=1e-3, eval_interval=100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for step in range(max_iters):
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % eval_interval == 0 or step == max_iters - 1:
            losses = estimate_loss(model, eval_iters=50)
            print(f'step {step:4d} | train loss {losses["train"]:.4f} | val loss {losses["val"]:.4f}')

train_model(bigram_model, max_iters=10000, lr=1e-3, eval_interval=100)

step    0 | train loss 4.7733 | val loss 4.7705
step  100 | train loss 4.6571 | val loss 4.6468
step  200 | train loss 4.5290 | val loss 4.5319
step  300 | train loss 4.4345 | val loss 4.4323
step  400 | train loss 4.3157 | val loss 4.3279
step  500 | train loss 4.2179 | val loss 4.2301
step  600 | train loss 4.1386 | val loss 4.1236
step  700 | train loss 4.0434 | val loss 4.0296
step  800 | train loss 3.9364 | val loss 3.9481
step  900 | train loss 3.8414 | val loss 3.8564
step 1000 | train loss 3.7797 | val loss 3.7716
step 1100 | train loss 3.6978 | val loss 3.7193
step 1200 | train loss 3.6420 | val loss 3.6370
step 1300 | train loss 3.5590 | val loss 3.5548
step 1400 | train loss 3.4752 | val loss 3.5093
step 1500 | train loss 3.4255 | val loss 3.4444
step 1600 | train loss 3.3595 | val loss 3.3712
step 1700 | train loss 3.3253 | val loss 3.3213
step 1800 | train loss 3.2500 | val loss 3.2569
step 1900 | train loss 3.2056 | val loss 3.2037
step 2000 | train loss 3.1391 | val loss

In [15]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
bigram_text = decode(bigram_model.generate(start, max_new_tokens=200)[0].tolist())
print('Bigram 生成样例：')
print(bigram_text)

Bigram 生成样例：

Ank m: got c

An bathors,
HEME:
Y harna sawqG resiusONDWArou ar curu?
WAh y wiver, t my irtst mangoulimelind.
Whe chatheibed.
QUSCont tsthimyshe
S: mbiSeanovonqu.
YAyous be
DT:
R:
YCE:
Hondd mand!

LY


## 二、三种不同的均值平均版本（因果前缀聚合）

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

# 版本1：双重循环，直接按定义求前缀均值
xbow1 = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]
        xbow1[b, t] = xprev.mean(dim=0)

# 版本2：下三角矩阵 + 行归一化 + 矩阵乘法
wei2 = torch.tril(torch.ones(T, T))
# print(wei2)
# print(wei2.sum(dim=1, keepdim=True))
wei2 = wei2 / wei2.sum(dim=1, keepdim=True)
xbow2 = wei2 @ x

# 版本3：masked softmax（把未来位置置为 -inf）
tril = torch.tril(torch.ones(T, T))
wei3 = torch.zeros((T, T))
wei3 = wei3.masked_fill(tril == 0, float('-inf'))
wei3 = F.softmax(wei3, dim=-1)
xbow3 = wei3 @ x

print('版本1与版本2是否一致:', torch.allclose(xbow1, xbow2, atol=1e-5))
print('版本1与版本3是否一致:', torch.allclose(xbow1, xbow3, atol=1e-5))

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.]])
版本1与版本2是否一致: True
版本1与版本3是否一致: True


## 三、Self-Attention 模型：训练与生成

In [19]:
class SelfAttentionLM(nn.Module):
    """
    一个最小可训练的自注意力语言模型（单头）：
    - token embedding + position embedding
    - 单头因果自注意力
    - 线性层投影到词表 logits
    """
    def __init__(self, vocab_size, n_embd=64, block_size=8):
        super().__init__()
        self.block_size = block_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(block_size, n_embd)

        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)

        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok = self.token_embedding(idx)
        pos = self.pos_embedding(torch.arange(T, device=idx.device))
        x = tok + pos

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # 注意力分数 + 缩放
        wei = (q @ k.transpose(-2, -1)) * (self.n_embd ** -0.5)

        # 因果 mask：当前位置不能看未来
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # 加权汇聚历史信息
        x = wei @ v

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 仅保留最近 block_size 个 token 作为上下文窗口
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits_last = logits[:, -1, :]
            probs = F.softmax(logits_last, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

attn_model = SelfAttentionLM(vocab_size=vocab_size, n_embd=64, block_size=block_size).to(device)
logits, loss = attn_model(xb, yb)
print('Self-Attention logits 形状:', logits.shape, 'loss:', float(loss))

Self-Attention logits 形状: torch.Size([256, 65]) loss: 4.210417747497559


In [21]:
train_model(attn_model, max_iters=10000, lr=3e-4, eval_interval=150)

start = torch.zeros((1, 1), dtype=torch.long, device=device)
attn_text = decode(attn_model.generate(start, max_new_tokens=200)[0].tolist())
print('Self-Attention 生成样例：')
print(attn_text)

step    0 | train loss 2.7574 | val loss 2.7650
step  150 | train loss 2.6727 | val loss 2.6611
step  300 | train loss 2.6307 | val loss 2.6226
step  450 | train loss 2.5690 | val loss 2.5892
step  600 | train loss 2.5491 | val loss 2.5439
step  750 | train loss 2.5535 | val loss 2.5313
step  900 | train loss 2.5027 | val loss 2.5227
step 1050 | train loss 2.4751 | val loss 2.5082
step 1200 | train loss 2.4829 | val loss 2.4911
step 1350 | train loss 2.4736 | val loss 2.4683
step 1500 | train loss 2.4742 | val loss 2.4577
step 1650 | train loss 2.4590 | val loss 2.4608
step 1800 | train loss 2.4587 | val loss 2.4290
step 1950 | train loss 2.4369 | val loss 2.4528
step 2100 | train loss 2.4187 | val loss 2.4539
step 2250 | train loss 2.3993 | val loss 2.4360
step 2400 | train loss 2.4192 | val loss 2.4080
step 2550 | train loss 2.4187 | val loss 2.4191
step 2700 | train loss 2.4390 | val loss 2.4619
step 2850 | train loss 2.4430 | val loss 2.4538
step 3000 | train loss 2.4271 | val loss